#Ollama:
Ollama lets you run LLMs locally on your machine (Mac/Linux/Windows) in a CLI-style chat interface.

🔁 Exported the Hugging Face fine-tuned model and convert it to GGUF or Ollama-compatible format using transformers + transformers-to-ollama.

In [ ]:
# Install dependencies
!pip install -q transformers datasets

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Load Model

In [ ]:
# ✅ Load model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# Load Dataset

In [ ]:
# ✅ Create synthetic mental health support dataset
data = {
    "text": [
        "User: I'm feeling really anxious lately.\nAssistant: I'm really sorry you're feeling this way. You're not alone, and I'm here for you.",
        "User: I don't know how to deal with stress.\nAssistant: It's okay to feel overwhelmed. Taking small steps like deep breathing can help.",
        "User: I feel like nobody understands me.\nAssistant: That must be really hard. But please know your feelings are valid and someone does care.",
        "User: I can't sleep because of my worries.\nAssistant: Sleep can be tough when the mind is racing. Sometimes journaling or calming music can help.",
    ]
}

In [ ]:
# Wrap in Hugging Face Dataset
dataset = Dataset.from_dict(data)

# Tokenize Dataset

In [ ]:
# Tokenize
def tokenize_fn(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_fn)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

# Setup Training

In [ ]:
# ✅ Training setup
training_args = TrainingArguments(
    output_dir="./mental-health-chatbot",
    per_device_train_batch_size=1,
    num_train_epochs=2,
    logging_steps=1,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

<ipython-input-20-369605fb9ae3>:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Model Finetuning

In [ ]:
# Fine-tune the model
trainer.train()

Step,Training Loss
1,4.262500
2,4.067400
3,2.924600
4,3.261900
5,2.539000
6,2.936400
7,3.073400
8,2.962400


TrainOutput(global_step=8, training_loss=3.253467172384262, metrics={'train_runtime': 25.6019, 'train_samples_per_second': 0.312, 'train_steps_per_second': 0.312, 'total_flos': 261296750592.0, 'train_loss': 3.253467172384262, 'epoch': 2.0})

# Inference - Chat With the Bot

In [ ]:
prompt = "User: I'm feeling very stressed at work.\nAssistant:"
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

# Set pad_token_id explicitly and add attention_mask
output = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,         # Enable sampling to avoid repetition
    top_k=50,                # Top-k sampling
    top_p=0.95,              # Nucleus sampling
    temperature=0.7          # Lower temp = less randomness
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


User: I'm feeling very stressed at work.
Assistant: You're a bit like a person.
Assistant: It's like I'm feeling really stressed at work.
Assistant: You're a bit worried about your work.
Assistant: I'm feeling very anxious at work.
Assistant: You're


Simulated fine-tuning a mental health support chatbot, teaching a model to respond empathetically using distilgpt2.

✅ Used prompt + high-quality assistant responses to teach supportive behavior.

✅ Adapted to Hugging Face + Transformers.

- References: https://medium.com/@mauryaanoop3/fine-tuning-microsoft-phi3-with-unsloth-for-mental-health-chatbot-development-ddea4e0c46e7

# Export to Ollama

In [ ]:
import os

checkpoints = [ckpt for ckpt in os.listdir("./mental-health-chatbot") if ckpt.startswith("checkpoint")]
print(checkpoints)


['checkpoint-8']


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./mental-health-chatbot/checkpoint-8"

model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)


In [ ]:
model.save_pretrained("./mental-health-model")
tokenizer.save_pretrained("./mental-health-model")


('./mental-health-model/tokenizer_config.json',
 './mental-health-model/special_tokens_map.json',
 './mental-health-model/vocab.json',
 './mental-health-model/merges.txt',
 './mental-health-model/added_tokens.json',
 './mental-health-model/tokenizer.json')

In [ ]:
!zip -r mental_health_model.zip ./mental-health-model
from google.colab import files
files.download("mental_health_model.zip")


  adding: mental-health-model/ (stored 0%)
  adding: mental-health-model/vocab.json (deflated 59%)
  adding: mental-health-model/tokenizer.json (deflated 82%)
  adding: mental-health-model/config.json (deflated 52%)
  adding: mental-health-model/model.safetensors (deflated 7%)
  adding: mental-health-model/merges.txt (deflated 53%)
  adding: mental-health-model/generation_config.json (deflated 24%)
  adding: mental-health-model/tokenizer_config.json (deflated 56%)
  adding: mental-health-model/special_tokens_map.json (deflated 80%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pip install transformers transformers-to-ollama

unzip unsloth_phi2_export.zip

transformers-to-ollama \
  --model ./unsloth_phi2_export \
  --output ./phi2-ollama \
  --license mit \
  --format gguf

ollama create phi2-chatbot -f ./phi2-ollama/Ollamafile
ollama run phi2-chatbot


In [ ]:
!jupyter nbconvert --ClearMetadataPreprocessor.enabled=True \
  --ClearMetadataPreprocessor.clear_cell_metadata=True \
  --ClearMetadataPreprocessor.clear_notebook_metadata=True \
  --to notebook \
  --output cleaned_notebook.ipynb \
  your_notebook.ipynb

